# 10 — ALTitude × DepMap ALT Merge

**First mutating notebook since the audit freeze.**

Integrates ALTitude-derived ALT calls from notebook 09 into the master manifest
(`alt_status` / `alt_source` / `source_confidence`) for the 31 RNA-backed OS
DepMap cell lines, under strict priority rules that never regress an existing call.

**Priority hierarchy (highest → lowest):**
| Priority | Source | Written as |
|---|---|---|
| 1 (kept) | Existing manifest gold (Taglialatela/Lawlor) | unchanged |
| 2 | ALTitude experimental (C-circle, APB, qPCR) | `alt_gold_positive` / `alt_negative_high_conf` |
| 2★ | HOS special case (ACH-000613, PMID:33344901) | `alt_negative_high_conf` |
| 3 | ALTitude prediction-only (ML model call) | `alt_silver` |
| — | No ALTitude call | unchanged (`alt_unknown`) |

**Hard invariants:** never downgrade, never stamp `peer_reviewed` onto `alt_unknown`, never overwrite gold with silver.

**Idempotent at the row level:** re-running produces zero further changes.

**Output:** updated `data/manifests/master_manifest.csv`, `data/processed/notebook_10_report.md`

In [1]:
import sys, random, time
from pathlib import Path

random.seed(42)
RUN_TIMESTAMP = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ — tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

from src.utils import log_versions
import pandas as pd

log_versions()
print(f'Run: {RUN_TIMESTAMP}')

ROOT: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline
Library versions:
  pandas: 3.0.2
  numpy: 2.4.4
  scipy: 1.17.1
  pyranges: 0.1.4
  openpyxl: 3.1.5
  requests: 2.33.1
  GEOparse: 2.0.4
Run: 2026-05-20T14:34:35Z


## 1 — Load manifest and shared DepMap lookups

In [2]:
from src.manifest_builder import load_altitude_cellines
from src.aim1_inventory import DEPMAP_RNA_PATH, depmap_rna_model_ids
from src.recovery_inventory import (
    MODEL_CSV_PATH,
    build_depmap_name_lookup,
    select_route_b_candidates,
    build_route_b_table,
)

MANIFEST_PATH = ROOT / 'data' / 'manifests' / 'master_manifest.csv'
manifest_pre = pd.read_csv(MANIFEST_PATH, dtype=str)
print(f'Manifest rows: {len(manifest_pre)}')

model_df = pd.read_csv(MODEL_CSV_PATH, dtype=str)
depmap_rna_models = depmap_rna_model_ids(DEPMAP_RNA_PATH)
altitude = load_altitude_cellines()
altitude_ids = set(altitude['sample_id'].astype(str))
print(f'Model.csv rows: {len(model_df)}')
print(f'DepMap RNA ModelIDs: {len(depmap_rna_models)}')
print(f'ALTitude panel rows: {len(altitude)}')

Manifest rows: 115
Model.csv rows: 2154
DepMap RNA ModelIDs: 1719
ALTitude panel rows: 959


## 2 — Build ALTitude-DepMap inventory (notebook 09 machinery)

Uses Route B to identify RNA-backed DepMap ModelIDs, then joins ALTitude
`ALT_pred_class` (not `Curated.Marker`) with experimental/prediction_only tiering.
Sanity assertions (U2OS/SAOS2/SJSA1/MG63) are enforced inside
`build_altitude_depmap_inventory` — will raise if column binding is wrong.

In [3]:
from src.altitude_depmap_inventory import build_altitude_depmap_inventory

lookup = build_depmap_name_lookup(model_df)
route_b_candidates = select_route_b_candidates(manifest_pre, altitude_ids)
route_b = build_route_b_table(route_b_candidates, lookup, model_df, depmap_rna_models)
rna_backed = route_b[route_b['in_depmap_rna']]
rna_model_ids: set[str] = set(rna_backed['mapped_model_id'].dropna().astype(str))

# build_altitude_depmap_inventory raises ValueError if ground-truth assertions fail.
inv = build_altitude_depmap_inventory(manifest_pre, altitude, model_df, rna_model_ids)
os_inv = inv[inv['os_status'] == 'osteosarcoma']

print(f'RNA-backed unique ModelIDs: {len(rna_model_ids)}')
print(f'Inventory rows (all):   {len(inv)}')
print(f'Inventory rows (OS):    {len(os_inv)}')
print()
print('altitude_evidence_tier (OS):')  
print(os_inv['altitude_evidence_tier'].value_counts().to_string())
print()
print('altitude_alt_status (OS):')
print(os_inv['altitude_alt_status'].value_counts(dropna=False).to_string())
print()
print('Notebook 09 sanity assertions: PASSED (no ValueError raised)')

RNA-backed unique ModelIDs: 32
Inventory rows (all):   32
Inventory rows (OS):    31

altitude_evidence_tier (OS):
altitude_evidence_tier
experimental       20
prediction_only    11

altitude_alt_status (OS):
altitude_alt_status
alt_gold_positive         12
alt_negative_high_conf     7
ambiguous_or_missing       6
alt_silver                 6

Notebook 09 sanity assertions: PASSED (no ValueError raised)


## 3 — Pre-merge state snapshot

In [4]:
print('=== PRE-MERGE: alt_status overall ===')
print(manifest_pre['alt_status'].value_counts(dropna=False).to_string())
print()
print('=== PRE-MERGE: alt_status by cohort ===')
print(pd.crosstab(manifest_pre['cohort'], manifest_pre['alt_status'], dropna=False).to_string())
print()
print('=== PRE-MERGE: include_aim2 counts ===')
for col in ('include_aim2_strict', 'include_aim2_pragmatic', 'include_aim2_broad'):
    n = (manifest_pre[col].astype(str).str.lower() == 'true').sum()
    print(f'  {col}: {n}')

=== PRE-MERGE: alt_status overall ===
alt_status
alt_gold_positive         60
alt_unknown               25
alt_negative_high_conf    19
alt_silver                11

=== PRE-MERGE: alt_status by cohort ===
alt_status          alt_gold_positive  alt_negative_high_conf  alt_silver  alt_unknown
cohort                                                                                
DepMap_26Q1                        12                       8          11            4
Lawlor_2026_GCCRI                  16                       1           0            0
PCGP_2014                          11                       2           0           21
Taglialatela_2026                   5                       2           0            0
de_Nonneville_2022                 16                       6           0            0

=== PRE-MERGE: include_aim2 counts ===
  include_aim2_strict: 79
  include_aim2_pragmatic: 90
  include_aim2_broad: 90


## 4 — Build merge plan

No manifest mutation yet. Shows exactly what will change.

In [5]:
from src.altitude_depmap_merge import (
    build_merge_plan,
    apply_merge_plan,
    assert_merge_invariants,
    assert_ground_truth,
    recompute_inclusion_flags,
    build_change_summary,
    HOS_ID,
)

plan = build_merge_plan(manifest_pre, inv)

writes   = [a for a in plan if a['action'] == 'write']
non_writes = [a for a in plan if a['action'] != 'write']

print(f'Total plan entries: {len(plan)}')
print(f'  → write:     {len(writes)}')
action_counts: dict = {}
for a in non_writes:
    action_counts[a['action']] = action_counts.get(a['action'], 0) + 1
for k, v in sorted(action_counts.items()):
    print(f'  → {k}: {v}')
print()

# Detail breakdown of writes by proposed call
from collections import Counter
write_types = Counter(w['proposed_alt'] for w in writes)
print('Writes by proposed alt_status:')
for status, count in write_types.most_common():
    print(f'  {status}: {count}')
print()

print('HOS special case:')
hos_action = next((a for a in plan if a['model_id'] == HOS_ID), None)
if hos_action:
    print(f'  action={hos_action["action"]}  proposed={hos_action["proposed_alt"]}  conf={hos_action["proposed_conf"]}')
    print(f'  source: {hos_action["proposed_src"]}')

Total plan entries: 32
  → write:     0
  → skip_non_os: 1
  → skip_same: 31

Writes by proposed alt_status:

HOS special case:
  action=skip_same  proposed=alt_negative_high_conf  conf=peer_reviewed
  source: ALTitude_2024; PMID:33344901; ALT_assay=C-circle; ALT_assay_strength=Negative (low C-circle signal)


## 5 — Show all planned writes

In [6]:
print(f'{'model_id':<15}  {'old_alt_status':<26}  {'new_alt_status':<26}  {'evidence_tier'}')
print('-' * 95)
for w in writes:
    # Get evidence tier from inventory
    inv_hit = inv[inv['model_id'] == w['model_id']]
    tier = inv_hit['altitude_evidence_tier'].iloc[0] if len(inv_hit) else 'special_case'
    if w['model_id'] == HOS_ID:
        tier = 'experimental (special case)'
    print(f"{w['model_id']:<15}  {w['current_alt']:<26}  {w['proposed_alt']:<26}  {tier}")

model_id         old_alt_status              new_alt_status              evidence_tier
-----------------------------------------------------------------------------------------------


## 6 — Apply merge plan to manifest copy

In [7]:
manifest_merged = apply_merge_plan(manifest_pre, plan)
print(f'Manifest rows: {len(manifest_merged)}  (unchanged — merge is row-level, not structural)')

Manifest rows: 115  (unchanged — merge is row-level, not structural)


## 7 — Assert invariants and ground-truth

Raises `ValueError` and aborts if any of the three hard invariants or four
ground-truth checks fail. **Do not modify the assertion cell.**

In [8]:
# Invariant assertions
assert_merge_invariants(manifest_pre, manifest_merged, plan)
print('I1 (no downgrade):             PASS')
print('I2 (no peer_reviewed/unknown): PASS')
print('I3 (no gold→silver):           PASS')

# Ground-truth assertions (U2OS, SAOS2, SJSA1, MG63)
assert_ground_truth(manifest_merged)
print()
GROUND_TRUTH = {
    'ACH-000364': ('U2OS',  'positive', 'alt_gold_positive'),
    'ACH-000410': ('SAOS2', 'positive', 'alt_gold_positive'),
    'ACH-000748': ('SJSA1', 'negative', 'alt_negative_high_conf'),
    'ACH-000359': ('MG63',  'negative', 'alt_negative_high_conf'),
}
for mid, (name, arm, expected_status) in GROUND_TRUTH.items():
    actual = manifest_merged[manifest_merged['sample_id']==mid]['alt_status'].iloc[0]
    print(f'  {mid} ({name}): {actual}  [{"PASS" if actual == expected_status else "FAIL"}')
print()
print('HOS (ACH-000613):')
hos_row = manifest_merged[manifest_merged['sample_id']==HOS_ID]
if len(hos_row):
    print(f'  alt_status={hos_row["alt_status"].iloc[0]}')
    print(f'  source_confidence={hos_row["source_confidence"].iloc[0]}')
    print(f'  alt_source={hos_row["alt_source"].iloc[0]}')

I1 (no downgrade):             PASS
I2 (no peer_reviewed/unknown): PASS
I3 (no gold→silver):           PASS

  ACH-000364 (U2OS): alt_gold_positive  [PASS
  ACH-000410 (SAOS2): alt_gold_positive  [PASS
  ACH-000748 (SJSA1): alt_negative_high_conf  [PASS
  ACH-000359 (MG63): alt_negative_high_conf  [PASS

HOS (ACH-000613):
  alt_status=alt_negative_high_conf
  source_confidence=peer_reviewed
  alt_source=ALTitude_2024; PMID:33344901; ALT_assay=C-circle; ALT_assay_strength=Negative (low C-circle signal)


## 8 — Re-derive include_aim2_* flags

In [9]:
manifest_final = recompute_inclusion_flags(manifest_merged)

print('=== include_aim2 before → after ===')
for col in ('include_aim2_strict', 'include_aim2_pragmatic', 'include_aim2_broad'):
    pre_n  = (manifest_pre[col].astype(str).str.lower() == 'true').sum()
    post_n = (manifest_final[col].astype(str).str.lower() == 'true').sum()
    delta  = post_n - pre_n
    print(f'  {col}: {pre_n} → {post_n}  (Δ+{delta})')
print()

# Show DepMap_26Q1 rows that newly enter include_aim2_strict
depmap_post = manifest_final[manifest_final['cohort']=='DepMap_26Q1']
new_strict = depmap_post[depmap_post['include_aim2_strict'].astype(str).str.lower()=='true']
print(f'DepMap_26Q1 rows now in include_aim2_strict ({len(new_strict)}):')
for _, r in new_strict.iterrows():
    print(f'  {r["sample_id"]}  {r["alt_status"]}  ({r["source_confidence"]})')

=== include_aim2 before → after ===
  include_aim2_strict: 79 → 79  (Δ+0)
  include_aim2_pragmatic: 90 → 90  (Δ+0)
  include_aim2_broad: 90 → 90  (Δ+0)

DepMap_26Q1 rows now in include_aim2_strict (20):
  ACH-000082  alt_gold_positive  (peer_reviewed)
  ACH-000359  alt_negative_high_conf  (peer_reviewed)
  ACH-000364  alt_gold_positive  (peer_reviewed)
  ACH-000410  alt_gold_positive  (peer_reviewed)
  ACH-000613  alt_negative_high_conf  (peer_reviewed)
  ACH-000748  alt_negative_high_conf  (peer_reviewed)
  ACH-001001  alt_negative_high_conf  (peer_reviewed)
  ACH-001526  alt_gold_positive  (peer_reviewed)
  ACH-001715  alt_gold_positive  (peer_reviewed)
  ACH-001814  alt_negative_high_conf  (peer_reviewed)
  ACH-002067  alt_gold_positive  (peer_reviewed)
  ACH-002069  alt_negative_high_conf  (peer_reviewed)
  ACH-002142  alt_negative_high_conf  (peer_reviewed)
  ACH-002178  alt_gold_positive  (peer_reviewed)
  ACH-002470  alt_negative_high_conf  (peer_reviewed)
  ACH-002471  alt_gold

## 9 — Post-merge state and change summary

In [10]:
print('=== POST-MERGE: alt_status overall ===')
print(manifest_final['alt_status'].value_counts(dropna=False).to_string())
print()
print('=== POST-MERGE: alt_status × cohort ===')
print(pd.crosstab(manifest_final['cohort'], manifest_final['alt_status'], dropna=False).to_string())
print()
print('=== POST-MERGE: source_confidence × alt_status (DepMap_26Q1) ===')
depmap_final = manifest_final[manifest_final['cohort']=='DepMap_26Q1']
print(pd.crosstab(depmap_final['source_confidence'], depmap_final['alt_status'], dropna=False).to_string())

=== POST-MERGE: alt_status overall ===
alt_status
alt_gold_positive         60
alt_unknown               25
alt_negative_high_conf    19
alt_silver                11

=== POST-MERGE: alt_status × cohort ===
alt_status          alt_gold_positive  alt_negative_high_conf  alt_silver  alt_unknown
cohort                                                                                
DepMap_26Q1                        12                       8          11            4
Lawlor_2026_GCCRI                  16                       1           0            0
PCGP_2014                          11                       2           0           21
Taglialatela_2026                   5                       2           0            0
de_Nonneville_2022                 16                       6           0            0

=== POST-MERGE: source_confidence × alt_status (DepMap_26Q1) ===
alt_status         alt_gold_positive  alt_negative_high_conf  alt_silver  alt_unknown
source_confidence               

## 10 — Write manifest

In [11]:
manifest_final.to_csv(MANIFEST_PATH, index=False)
print(f'Manifest written: {MANIFEST_PATH}  ({MANIFEST_PATH.stat().st_size:,} bytes)')
print(f'  Rows: {len(manifest_final)}  Columns: {len(manifest_final.columns)}')

Manifest written: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline/data/manifests/master_manifest.csv  (64,029 bytes)
  Rows: 115  Columns: 46


## 11 — Write `data/processed/notebook_10_report.md`

In [12]:
REPORT_PATH = ROOT / 'data' / 'processed' / 'notebook_10_report.md'
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

change_summary = build_change_summary(manifest_pre, manifest_final, plan)

report_lines: list[str] = [
    '# Notebook 10 — ALTitude × DepMap ALT Merge\n',
    f'_Generated: {RUN_TIMESTAMP}_\n',
    '\n',
    'Integrates ALTitude curated ALT calls into the manifest for RNA-backed '
    'DepMap OS cell lines. First mutating notebook since the audit freeze.\n',
    '\n',
    '## Priority hierarchy\n',
    '| Tier | Source | alt_status written | source_confidence |\n',
    '|---|---|---|---|\n',
    '| 1 (kept) | Existing manifest gold | unchanged | unchanged |\n',
    '| 2 | ALTitude experimental | alt_gold_positive / alt_negative_high_conf | peer_reviewed |\n',
    '| 2★ | HOS special case (PMID:33344901) | alt_negative_high_conf | peer_reviewed |\n',
    '| 3 | ALTitude prediction-only | alt_silver | computational |\n',
    '| — | No call | unchanged (alt_unknown) | unchanged |\n',
    '\n',
    '## Invariant checks\n',
    '- I1 (no downgrade):             **PASS**\n',
    '- I2 (no peer_reviewed/unknown): **PASS**\n',
    '- I3 (no gold→silver):           **PASS**\n',
    '\n',
    '## Ground-truth checks\n',
]

GROUND_TRUTH = {
    'ACH-000364': ('U2OS',  'alt_gold_positive'),
    'ACH-000410': ('SAOS2', 'alt_gold_positive'),
    'ACH-000748': ('SJSA1', 'alt_negative_high_conf'),
    'ACH-000359': ('MG63',  'alt_negative_high_conf'),
}
for mid, (name, expected) in GROUND_TRUTH.items():
    actual = manifest_final[manifest_final['sample_id']==mid]['alt_status'].iloc[0]
    status = 'PASS' if actual == expected else 'FAIL'
    report_lines.append(f'- {mid} ({name}): `{actual}` [{status}]\n')

hos_row = manifest_final[manifest_final['sample_id']==HOS_ID]
hos_status = hos_row['alt_status'].iloc[0] if len(hos_row) else 'NOT FOUND'
report_lines.append(
    f'- ACH-000613 (HOS, special case): `{hos_status}` '
    '[NaN pred_class; C-circle negative; PMID:33344901]\n'
)
report_lines.append('\n')

report_lines.append(change_summary)

report_lines += [
    '## Idempotence\n',
    'Second-pass writes (must be 0): see cell below.\n',
    '\n',
    '## Next steps\n',
    '1. Re-run notebooks 05 (Pass A) and 04 (Pass B) to update audit tables.\n',
    '2. No DE or modeling code has been written.\n',
]

report_text = ''.join(report_lines)
REPORT_PATH.write_text(report_text)
print(f'Wrote {REPORT_PATH.relative_to(ROOT)}  ({REPORT_PATH.stat().st_size} bytes)')

Wrote data/processed/notebook_10_report.md  (2710 bytes)


## 12 — Idempotence check

Re-read the manifest just written and build a second merge plan.
All actions must be `skip_*` — zero further writes.

In [13]:
manifest_reread = pd.read_csv(MANIFEST_PATH, dtype=str)
plan2 = build_merge_plan(manifest_reread, inv)
writes2 = [a for a in plan2 if a['action'] == 'write']

assert len(writes2) == 0, (
    f'Idempotence FAILED: {len(writes2)} write(s) on second pass — '
    f'{[w["model_id"] for w in writes2]}'
)
print(f'Idempotence check PASSED: 0 writes on re-run.')

action_counts2 = {}
for a in plan2:
    action_counts2[a['action']] = action_counts2.get(a['action'], 0) + 1
for k, v in sorted(action_counts2.items()):
    print(f'  {k}: {v}')

Idempotence check PASSED: 0 writes on re-run.
  skip_non_os: 1
  skip_same: 31


In [14]:
import pandas as pd
from pathlib import Path

ROOT = Path('.').resolve()
ROOT = ROOT if (ROOT / 'src').is_dir() else ROOT.parent

# Load the post-merge manifest
manifest = pd.read_csv(ROOT / 'data' / 'manifests' / 'master_manifest.csv', dtype=str)

# Pull all alt_silver rows
silver = manifest[manifest['alt_status'] == 'alt_silver'][
    ['sample_id', 'cohort', 'alt_status', 'alt_source', 'source_confidence']
].copy()

print(f'alt_silver rows in manifest: {len(silver)}')
print(silver.to_string(index=False))
print()

# Now cross-reference with ALTitude to recover the ALT_pred_class
from src.manifest_builder import load_altitude_cellines
altitude = load_altitude_cellines()
# ALTitude's sample_id is the DepMap ModelID (Achilles column from the raw table)
silver_with_call = silver.merge(
    altitude[['sample_id', 'alt_status']].rename(
        columns={'alt_status': 'altitude_alt_status_resolved'}
    ),
    on='sample_id', how='left'
)
print('Silver lines × ALTitude resolved status:')
print(silver_with_call[['sample_id','altitude_alt_status_resolved']].to_string(index=False))

# And: how does compute_inclusion_flags treat alt_silver?
import inspect
from src.label_rules import compute_inclusion_flags
print('\n=== compute_inclusion_flags source ===')
print(inspect.getsource(compute_inclusion_flags))

alt_silver rows in manifest: 11
 sample_id      cohort alt_status              alt_source source_confidence
ACH-001275 DepMap_26Q1 alt_silver ALTitude_2024_predicted     computational
ACH-001813 DepMap_26Q1 alt_silver ALTitude_2024_predicted     computational
ACH-001817 DepMap_26Q1 alt_silver ALTitude_2024_predicted     computational
ACH-002433 DepMap_26Q1 alt_silver ALTitude_2024_predicted     computational
ACH-002779 DepMap_26Q1 alt_silver ALTitude_2024_predicted     computational
ACH-003171 DepMap_26Q1 alt_silver ALTitude_2024_predicted     computational
ACH-003176 DepMap_26Q1 alt_silver ALTitude_2024_predicted     computational
ACH-003178 DepMap_26Q1 alt_silver ALTitude_2024_predicted     computational
ACH-003185 DepMap_26Q1 alt_silver ALTitude_2024_predicted     computational
ACH-003573 DepMap_26Q1 alt_silver ALTitude_2024_predicted     computational
ACH-003587 DepMap_26Q1 alt_silver ALTitude_2024_predicted     computational

Silver lines × ALTitude resolved status:
 sample_id alt

In [15]:
import pandas as pd
from pathlib import Path

ROOT = Path('.').resolve()
ROOT = ROOT if (ROOT / 'src').is_dir() else ROOT.parent

manifest = pd.read_csv(ROOT / 'data' / 'manifests' / 'master_manifest.csv', dtype=str)

# 1. List all columns and verify the new one
print(f'Total columns: {len(manifest.columns)}')
print()
print('All column names:')
for c in manifest.columns:
    print(f'  {c}')

# 2. Check whether alt_call_direction exists
print()
if 'alt_call_direction' in manifest.columns:
    print('=== alt_call_direction × alt_status crosstab ===')
    print(pd.crosstab(manifest['alt_status'], manifest['alt_call_direction'], dropna=False).to_string())
    print()
    print('=== silver lines: alt_call_direction values ===')
    silver = manifest[manifest['alt_status'] == 'alt_silver']
    print(silver[['sample_id', 'alt_status', 'alt_call_direction']].to_string(index=False))
else:
    print('!! alt_call_direction column does NOT exist in manifest. Fix did not land.')

# 3. Check whether loader exposes altitude_pred_class
print()
from src.manifest_builder import load_altitude_cellines
altitude = load_altitude_cellines()
print(f'load_altitude_cellines columns: {list(altitude.columns)}')
if 'altitude_pred_class' in altitude.columns:
    silver_in_alt = altitude[altitude['alt_status'] == 'alt_silver']
    print()
    print(f'Silver rows in loader output: {len(silver_in_alt)}')
    print('Distribution of altitude_pred_class for silver:')
    print(silver_in_alt['altitude_pred_class'].value_counts(dropna=False).to_string())
else:
    print('!! altitude_pred_class column NOT in loader output. Partial fix only.')

# 4. Check whether derive_alt_call_direction helper exists
print()
try:
    from src.manifest_builder import derive_alt_call_direction
    print('derive_alt_call_direction: AVAILABLE')
    # Test it
    print(f"  ('alt_gold_positive', None) -> {derive_alt_call_direction('alt_gold_positive', None)}")
    print(f"  ('alt_silver', 'ALT+')      -> {derive_alt_call_direction('alt_silver', 'ALT+')}")
    print(f"  ('alt_silver', 'ALT-')      -> {derive_alt_call_direction('alt_silver', 'ALT-')}")
    print(f"  ('alt_unknown', None)        -> {derive_alt_call_direction('alt_unknown', None)}")
except ImportError:
    try:
        from src.label_rules import derive_alt_call_direction
        print('derive_alt_call_direction: AVAILABLE (in label_rules)')
    except ImportError:
        print('!! derive_alt_call_direction helper NOT found. Fix did not land.')

Total columns: 46

All column names:
  sample_id
  cohort
  sample_type
  has_rna
  has_wes
  has_wgs
  has_sc
  has_clinical
  tp53_status
  tp53_evidence
  tcab1_rna_quantile
  tcab1_rna_value
  tcab1_rna_quantile_within_cohort
  tcab1_genomic_status
  tcab1_genomic_evidence
  tcab1_functional_status
  tcab1_functional_evidence
  tcab1_overall_status
  tcab1_confidence
  sv_architecture_17p13_1
  sv_architecture_evidence
  alt_status
  alt_source
  source_confidence
  alt_assay_evidence
  telomerase_status
  tert_expression
  telomerase_evidence
  atrx_status
  daxx_status
  smarcal1_status
  top3a_status
  purity_estimate
  purity_quartile
  include_aim1_primary_strict
  include_aim1_primary_pragmatic
  include_aim1_primary_broad
  include_aim2_strict
  include_aim2_pragmatic
  include_aim2_broad
  include_tp53_sensitivity
  include_telomerase_negative_secondary
  notes
  sv_architecture_auto
  sv_auto_reason
  alt_call_direction

=== alt_call_direction × alt_status crosstab ===
alt